In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00057182/sub-A00057182_ses-NFB3_T1w_brain.nii
/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00057182/sub-A00057182_ses-NFB3_T1w_brainmask.nii
/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00057182/sub-A00057182_ses-NFB3_T1w.nii
/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00056097/sub-A00056097_ses-NFB3_T1w.nii
/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00056097/sub-A00056097_ses-NFB3_T1w_brainmask.nii
/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00056097/sub-A00056097_ses-NFB3_T1w_brain.nii
/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00061806/sub-A00061806_ses-NFB3_T1w_brain.nii
/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00061806/sub-A00061806_ses-NFB3_T1w_brainmask.nii
/kaggle/input/datasets/shresthapriya/skull-stripped/NFBS_Dataset/A00061806/sub-A00061806_ses-NFB3_T1w.ni

In [2]:
!pip -q install nibabel scipy pandas

In [3]:
from pathlib import Path
import re, zipfile
import nibabel as nib
import numpy as np
import pandas as pd
from scipy.ndimage import zoom
from nibabel.processing import resample_from_to

TARGET_SHAPE = (96, 96, 96)

ISLES_ROOT = Path("/kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset")
NFBS_ROOT = Path("/kaggle/input/datasets/shresthapriya/skull-stripped")

print("ISLES path:", ISLES_ROOT, "| exists:", ISLES_ROOT.exists())
print("NFBS path :", NFBS_ROOT, "| exists:", NFBS_ROOT.exists())

if not ISLES_ROOT.exists() or not NFBS_ROOT.exists():
    print("\nFolders available under /kaggle/input:")
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir():
            print(p)
    raise FileNotFoundError("A dataset path was not found. Compare the printed paths with your Kaggle input paths.")

def is_nifti(p):
    name = p.name.lower()
    return name.endswith(".nii") or name.endswith(".nii.gz")

def all_nifti(root):
    return sorted(p for p in root.rglob("*") if p.is_file() and is_nifti(p))

all_isles = all_nifti(ISLES_ROOT)
all_nfbs = all_nifti(NFBS_ROOT)

print("\nAll ISLES NIfTI files:", len(all_isles))
print("All NFBS NIfTI files :", len(all_nfbs))
print("\nISLES examples:")
for p in all_isles[:10]: print(p)
print("\nNFBS examples:")
for p in all_nfbs[:10]: print(p)

if not all_nfbs:
    raise ValueError("No .nii/.nii.gz files found in NFBS. Check whether files are archived or use another format.")
if not all_isles:
    raise ValueError("No .nii/.nii.gz files found in ISLES. Check whether files are archived or use another format.")

ISLES path: /kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset | exists: True
NFBS path : /kaggle/input/datasets/shresthapriya/skull-stripped | exists: True

All ISLES NIfTI files: 1000
All NFBS NIfTI files : 375

ISLES examples:
/kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset/ISLES-2022/ISLES-2022/derivatives/sub-strokecase0001/ses-0001/sub-strokecase0001_ses-0001_msk.nii
/kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset/ISLES-2022/ISLES-2022/derivatives/sub-strokecase0002/ses-0001/sub-strokecase0002_ses-0001_msk.nii
/kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset/ISLES-2022/ISLES-2022/derivatives/sub-strokecase0003/ses-0001/sub-strokecase0003_ses-0001_msk.nii
/kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset/ISLES-2022/ISLES-2022/derivatives/sub-strokecase0004/ses-0001/sub-strokecase0004_ses-0001_msk.nii
/kaggle/input/datasets/orvile/isles-2022-brain-stoke-dataset/ISLES-2022/ISLES-2022/derivatives/sub-strokecase0005/ses-0001/s

In [4]:
def path_text(p):
    return (str(p.parent) + "/" + p.name).lower()

def looks_like_mask(p):
    s = path_text(p)
    return any(token in s for token in (
        "_msk", "mask", "segmentation", "lesion_mask", "/masks/", "/mask/", "label"
    ))

nfbs_files = [
    p for p in all_nfbs
    if not looks_like_mask(p)
    and not any(token in path_text(p) for token in ("brainmask", "brain_mask", "probmask"))
]

isles_masks = [p for p in all_isles if looks_like_mask(p)]

isles_flair = [
    p for p in all_isles
    if "flair" in path_text(p)
    and not looks_like_mask(p)
    and not any(token in path_text(p) for token in ("dwi", "adc"))
]

print("NFBS MRI candidates:", len(nfbs_files))
print("ISLES FLAIR candidates:", len(isles_flair))
print("ISLES mask candidates:", len(isles_masks))

if not nfbs_files:
    raise ValueError("No NFBS MRI candidates after excluding mask-like files. Review the printed NFBS examples.")
if not isles_flair:
    print("\nISLES NIfTI paths for inspection:")
    for p in all_isles[:100]: print(p)
    raise ValueError("No path containing 'FLAIR' found. Inspect the paths above and update the isles_flair selection to match your dataset's actual FLAIR filenames.")

NFBS MRI candidates: 250
ISLES FLAIR candidates: 250
ISLES mask candidates: 250


In [5]:
OUT = Path("/kaggle/working/preprocessed_3d_nii")
NORMAL_OUT = OUT / "Normal"
STROKE_OUT = OUT / "Stroke"
MASK_OUT = OUT / "Stroke_lesion_masks"
for folder in (NORMAL_OUT, STROKE_OUT, MASK_OUT):
    folder.mkdir(parents=True, exist_ok=True)

def load_3d(path):
    img = nib.as_closest_canonical(nib.load(str(path)))
    data = img.get_fdata(dtype=np.float32)
    if data.ndim > 3:
        data = data[..., 0]
    if data.ndim != 3:
        raise ValueError(f"Expected 3D image; got shape {data.shape}: {path}")
    return img, np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

def get_foreground_bbox(data):
    vals = data[np.isfinite(data)]
    if vals.size == 0:
        return tuple(slice(0, n) for n in data.shape)
    threshold = max(float(np.percentile(vals, 10)), 0.0)
    fg = data > threshold
    if fg.sum() < 100:
        fg = data != 0
    coords = np.where(fg)
    if len(coords[0]) == 0:
        return tuple(slice(0, n) for n in data.shape)
    starts, ends = [], []
    for axis, n in enumerate(data.shape):
        margin = max(2, int(0.03 * n))
        starts.append(max(0, int(coords[axis].min()) - margin))
        ends.append(min(n, int(coords[axis].max()) + 1 + margin))
    return tuple(slice(starts[i], ends[i]) for i in range(3))

def resize_volume(data, order):
    factors = [TARGET_SHAPE[i] / data.shape[i] for i in range(3)]
    resized = zoom(data, factors, order=order, mode="nearest", prefilter=(order > 1))
    if resized.shape != TARGET_SHAPE:
        fixed = np.zeros(TARGET_SHAPE, dtype=resized.dtype)
        slices = tuple(slice(0, min(resized.shape[i], TARGET_SHAPE[i])) for i in range(3))
        fixed[slices] = resized[slices]
        resized = fixed
    return resized

def resized_affine(source_affine, crop_slices, cropped_shape):
    translation = np.eye(4)
    translation[:3, 3] = [sl.start or 0 for sl in crop_slices]
    scale = np.eye(4)
    for i in range(3):
        scale[i, i] = cropped_shape[i] / TARGET_SHAPE[i]
    return source_affine @ translation @ scale

def preprocess_mri(src, dst):
    img, data = load_3d(src)
    crop = get_foreground_bbox(data)
    cropped = data[crop]
    low, high = np.percentile(cropped, [1, 99])
    if high > low:
        cropped = (np.clip(cropped, low, high) - low) / (high - low)
    else:
        cropped = np.zeros_like(cropped, dtype=np.float32)
    resized = resize_volume(cropped.astype(np.float32), order=1).astype(np.float32)
    affine = resized_affine(img.affine, crop, cropped.shape)
    header = nib.Nifti1Header()
    header.set_data_dtype(np.float32)
    nib.save(nib.Nifti1Image(resized, affine, header=header), str(dst))
    return tuple(data.shape)

def preprocess_mask(mask_src, flair_src, dst):
    flair_img, flair_data = load_3d(flair_src)
    mask_img = nib.as_closest_canonical(nib.load(str(mask_src)))
    mask_data0 = mask_img.get_fdata(dtype=np.float32)
    if mask_data0.ndim > 3:
        mask_data0 = mask_data0[..., 0]
        mask_img = nib.Nifti1Image(mask_data0, mask_img.affine, mask_img.header)
    aligned_img = resample_from_to(mask_img, (flair_img.shape[:3], flair_img.affine), order=0)
    mask_data = (aligned_img.get_fdata() > 0).astype(np.uint8)
    crop = get_foreground_bbox(flair_data)
    cropped = mask_data[crop]
    resized = (resize_volume(cropped, order=0) > 0).astype(np.uint8)
    affine = resized_affine(flair_img.affine, crop, cropped.shape)
    header = nib.Nifti1Header()
    header.set_data_dtype(np.uint8)
    nib.save(nib.Nifti1Image(resized, affine, header=header), str(dst))
    return int(resized.sum())

def case_key(path):
    name = path.name.lower().replace(".nii.gz", "").replace(".nii", "")
    name = re.sub(r"(flair|dwi|adc|t1w|t2w|_msk|mask|seg|lesion|label)", "", name)
    return re.sub(r"[^a-z0-9]", "", name)

def find_matching_mask(flair_path, masks):
    key = case_key(flair_path)
    matches = [
        m for m in masks
        if case_key(m) == key
        or (case_key(m) and key and (case_key(m) in key or key in case_key(m)))
    ]
    return matches[0] if len(matches) == 1 else None

In [6]:
manifest, errors = [], []

for index, src in enumerate(nfbs_files, start=1):
    try:
        dst = NORMAL_OUT / f"normal_{index:04d}.nii"
        original_shape = preprocess_mri(src, dst)
        manifest.append({
            "sample_id": dst.stem,
            "label": "Normal",
            "modality": "T1w_NFBS",
            "source_file": str(src),
            "output_file": str(dst),
            "mask_file": "",
            "original_shape": str(original_shape),
            "processed_shape": str(TARGET_SHAPE)
        })
        if index % 10 == 0 or index == len(nfbs_files):
            print(f"NFBS processed {index}/{len(nfbs_files)}")
    except Exception as exc:
        errors.append({"source_file": str(src), "stage": "NFBS", "error": repr(exc)})
        print("NFBS skipped:", src, repr(exc))

NFBS processed 10/250
NFBS processed 20/250
NFBS processed 30/250
NFBS processed 40/250
NFBS processed 50/250
NFBS processed 60/250
NFBS processed 70/250
NFBS processed 80/250
NFBS processed 90/250
NFBS processed 100/250
NFBS processed 110/250
NFBS processed 120/250
NFBS processed 130/250
NFBS processed 140/250
NFBS processed 150/250
NFBS processed 160/250
NFBS processed 170/250
NFBS processed 180/250
NFBS processed 190/250
NFBS processed 200/250
NFBS processed 210/250
NFBS processed 220/250
NFBS processed 230/250
NFBS processed 240/250
NFBS processed 250/250


In [7]:
for index, src in enumerate(isles_flair, start=1):
    try:
        dst = STROKE_OUT / f"stroke_{index:04d}.nii"
        original_shape = preprocess_mri(src, dst)
        mask_src = find_matching_mask(src, isles_masks)
        mask_dst = ""
        lesion_voxels = None
        if mask_src is not None:
            mask_out = MASK_OUT / f"stroke_{index:04d}_lesion_mask.nii"
            lesion_voxels = preprocess_mask(mask_src, src, mask_out)
            mask_dst = str(mask_out)
            print("Mask match:", src.name, "->", mask_src.name)
        else:
            print("WARNING: no unique mask match for", src.name)

        manifest.append({
            "sample_id": dst.stem,
            "label": "Stroke",
            "modality": "FLAIR_ISLES2022",
            "source_file": str(src),
            "output_file": str(dst),
            "mask_file": mask_dst,
            "original_shape": str(original_shape),
            "processed_shape": str(TARGET_SHAPE),
            "lesion_voxels_after_resize": lesion_voxels
        })
        if index % 10 == 0 or index == len(isles_flair):
            print(f"ISLES processed {index}/{len(isles_flair)}")
    except Exception as exc:
        errors.append({"source_file": str(src), "stage": "ISLES", "error": repr(exc)})
        print("ISLES skipped:", src, repr(exc))

pd.DataFrame(manifest).to_csv(OUT / "preprocessing_manifest.csv", index=False)
pd.DataFrame(errors, columns=["source_file", "stage", "error"]).to_csv(OUT / "preprocessing_errors.csv", index=False)

print("Processed NIfTI files:", len(list(OUT.rglob("*.nii"))))
display(pd.DataFrame(manifest).head(10))

Mask match: sub-strokecase0001_ses-0001_FLAIR.nii -> sub-strokecase0001_ses-0001_msk.nii
Mask match: sub-strokecase0002_ses-0001_FLAIR.nii -> sub-strokecase0002_ses-0001_msk.nii
Mask match: sub-strokecase0003_ses-0001_FLAIR.nii -> sub-strokecase0003_ses-0001_msk.nii
Mask match: sub-strokecase0004_ses-0001_FLAIR.nii -> sub-strokecase0004_ses-0001_msk.nii
Mask match: sub-strokecase0005_ses-0001_FLAIR.nii -> sub-strokecase0005_ses-0001_msk.nii
Mask match: sub-strokecase0006_ses-0001_FLAIR.nii -> sub-strokecase0006_ses-0001_msk.nii
Mask match: sub-strokecase0007_ses-0001_FLAIR.nii -> sub-strokecase0007_ses-0001_msk.nii
Mask match: sub-strokecase0008_ses-0001_FLAIR.nii -> sub-strokecase0008_ses-0001_msk.nii
Mask match: sub-strokecase0009_ses-0001_FLAIR.nii -> sub-strokecase0009_ses-0001_msk.nii
Mask match: sub-strokecase0010_ses-0001_FLAIR.nii -> sub-strokecase0010_ses-0001_msk.nii
ISLES processed 10/250
Mask match: sub-strokecase0011_ses-0001_FLAIR.nii -> sub-strokecase0011_ses-0001_msk.ni

,sample_id,label,modality,source_file,output_file,mask_file,original_shape,processed_shape,lesion_voxels_after_resize
0,normal_0001,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
1,normal_0002,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
2,normal_0003,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
3,normal_0004,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
4,normal_0005,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
5,normal_0006,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
6,normal_0007,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
7,normal_0008,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
8,normal_0009,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN
9,normal_0010,Normal,T1w_NFBS,/kaggle/input/datasets/shresthapriya/skull-str...,/kaggle/working/preprocessed_3d_nii/Normal/nor...,,"(192, 256, 256)","(96, 96, 96)",NaN


In [8]:
checks = []
for path in sorted(OUT.rglob("*.nii")):
    image = nib.load(str(path))
    array = np.asanyarray(image.dataobj)
    checks.append({
        "file": str(path.relative_to(OUT)),
        "shape": str(image.shape),
        "dtype": str(array.dtype),
        "valid_shape": tuple(image.shape) == TARGET_SHAPE
    })

check_df = pd.DataFrame(checks)
display(check_df.head(20))
print("All saved NIfTI volumes are 96×96×96:",
      bool(check_df["valid_shape"].all()) if len(check_df) else False)

zip_path = Path("/kaggle/working/preprocessed_3d_nii.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(Path("preprocessed_3d_nii") / path.relative_to(OUT)))

print("ZIP created:", zip_path)
print("ZIP size (GB):", round(zip_path.stat().st_size / (1024**3), 3))
from IPython.display import FileLink, display
display(FileLink(str(zip_path)))

,file,shape,dtype,valid_shape
0,Normal/normal_0001.nii,"(96, 96, 96)",float32,True
1,Normal/normal_0002.nii,"(96, 96, 96)",float32,True
2,Normal/normal_0003.nii,"(96, 96, 96)",float32,True
3,Normal/normal_0004.nii,"(96, 96, 96)",float32,True
4,Normal/normal_0005.nii,"(96, 96, 96)",float32,True
5,Normal/normal_0006.nii,"(96, 96, 96)",float32,True
6,Normal/normal_0007.nii,"(96, 96, 96)",float32,True
7,Normal/normal_0008.nii,"(96, 96, 96)",float32,True
8,Normal/normal_0009.nii,"(96, 96, 96)",float32,True
9,Normal/normal_0010.nii,"(96, 96, 96)",float32,True


All saved NIfTI volumes are 96×96×96: True
ZIP created: /kaggle/working/preprocessed_3d_nii.zip
ZIP size (GB): 0.719


/kaggle/working/preprocessed_3d_nii.zip

In [9]:
import os
from IPython.display import FileLink, display

p = "/kaggle/working/preprocessed_3d_nii.zip"
print("exists:", os.path.exists(p))
if os.path.exists(p):
    print(round(os.path.getsize(p) / 1e6), "MB")
    display(FileLink("preprocessed_3d_nii.zip"))   # click the link it prints

exists: True
772 MB


/kaggle/working/preprocessed_3d_nii.zip

In [10]:
import os, subprocess, zipfile, shutil

URL = "https://www.dropbox.com/scl/fo/i51xt63roognvt7vuslbl/AG99uZljziHss5zJz4HiFis?rlkey=9w55le6tslwxlfz6c0viylmjb&st=b9cnvwh8&dl=1"
os.makedirs("/kaggle/working/ckpt", exist_ok=True)
subprocess.run(["curl", "-L", "-sS", "--retry", "3", "-o", "/tmp/brainiac_folder.zip", URL], check=True)

with zipfile.ZipFile("/tmp/brainiac_folder.zip") as z:
    names = z.namelist(); print(names)                       # you will see the other checkpoints here too
    hit = [n for n in names if os.path.basename(n) == "BrainIAC.ckpt"]
    assert hit, "BrainIAC.ckpt not in the zip: check the printed names"
    with z.open(hit[0]) as src, open("/kaggle/working/ckpt/BrainIAC.ckpt", "wb") as dst:
        shutil.copyfileobj(src, dst)
os.remove("/tmp/brainiac_folder.zip")
print("saved:", round(os.path.getsize("/kaggle/working/ckpt/BrainIAC.ckpt") / 1e6), "MB")

['/', 'idh.ckpt', 'stroke.ckpt', 'vit_mci.ckpt', 'BrainIAC.ckpt', 'brainage.ckpt', 'segmentation.ckpt', 'OS_ViT_BrainIAC.ckpt', 'sequence_classifcation.ckpt']
saved: 362 MB


In [11]:
# =====================================================================================
# BrainIAC fine-tuning: Normal / MCI / Alzheimer's / Stroke (4-class)
# 5-fold subject-grouped CV  ->  metrics + graphs  ->  final model  ->  ONNX package for Jetson
# Works as ONE Kaggle notebook cell (GPU on) or as `python brainiac_multiclass.py`.
# =====================================================================================
import os, sys, glob, json, re, time, random, shutil, subprocess, textwrap, copy, math, warnings
warnings.filterwarnings("ignore")

# ============================== CONFIG (edit only this block) =========================
CLASS_DIRS = {                      # one Kaggle dataset per class (searched recursively for .nii / .nii.gz)
    "normal":     "/kaggle/input/datasets/maharghakundu/normal-preprossed",
    "mci":        "/kaggle/input/datasets/maharghakundu/mci-preprossed",
    "alzheimers": "/kaggle/input/datasets/maharghakundu/alzeimer-preprossed",
    "stroke":     "/kaggle/input/datasets/maharghakundu/stroke-preprossed",
}
DATA_ROOT = os.environ.get("DATA_ROOT", "")     # alternative to CLASS_DIRS: a folder with labels.csv (path,label[,subject,age]) or class sub-folders
WITHIN_SOURCE = ["normal", "mci", "alzheimers"]  # classes that come from ONE dataset (OASIS-1): reported separately, free of dataset shortcuts
CKPT          = os.environ.get("CKPT") or None          # path to BrainIAC.ckpt; None = search for it
OUT           = os.environ.get("OUT", "/kaggle/working/results_mc")
CLASSES       = ["normal", "mci", "alzheimers", "stroke"]   # order = output index of the model
MODES         = ["full"]            # "full" = fine-tune everything. ["linear","full"] also compares head-only (~2x time)
EPOCHS        = {"full": 40, "linear": 60}
FOLDS         = 5
BATCH         = 6
LR_HEAD, LR_BACKBONE = 1e-3, 1e-4
SUBJECT_REGEX = [r"(OAS1_\d+)", r"(sub-[A-Za-z0-9]+)"]   # scans of one person must share a subject id (no train/test leakage)
SAMPLE_SCAN_IN_PACKAGE = True       # False if the teammate must not receive a raw dataset scan
SEED          = 42
LAYERS        = int(os.environ.get("LAYERS", 12))       # 12 = BrainIAC ViT-B. Only change for the CPU self-test.
SMOKE         = os.environ.get("SMOKE") == "1"          # tiny CPU self-test, leave off
if SMOKE: EPOCHS, FOLDS, BATCH, MODES = {"full": 2, "linear": 2}, 2, 4, ["full"]
# =======================================================================================

def pipi(*pk): subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *pk], check=False)
try:
    import monai, einops, nibabel, onnx, onnxruntime
except ImportError:
    pipi("monai", "einops", "nibabel", "onnx", "onnxruntime")

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, nibabel as nib
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix,
                             precision_recall_fscore_support, roc_curve)
from monai.networks.nets import ViT
from monai.transforms import Compose, RandAffine, RandFlip, RandScaleIntensity, RandShiftIntensity, RandGaussianNoise
from monai.utils import set_determinism

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); set_determinism(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
C = len(CLASSES)
os.makedirs(OUT, exist_ok=True)
for sub in ("plots", "weights", "deploy"): os.makedirs(f"{OUT}/{sub}", exist_ok=True)
print(f"device: {DEV}" + (f" ({torch.cuda.get_device_name(0)})" if DEV == "cuda" else "  <- no GPU: only OK for SMOKE=1"))

# ------------------------------------ checkpoint -------------------------------------
def find_ckpt():
    if CKPT and os.path.exists(CKPT): return CKPT
    for pat in ["/kaggle/input/**/BrainIAC.ckpt", "/kaggle/working/**/BrainIAC.ckpt", "/tmp/ckpt/BrainIAC.ckpt", "./**/BrainIAC.ckpt"]:
        h = glob.glob(pat, recursive=True)
        if h: return h[0]
    return None
CKPT_PATH = find_ckpt()
if CKPT_PATH is None and not SMOKE:
    raise SystemExit("BrainIAC.ckpt not found. Set CKPT=... or attach it as a Kaggle dataset. Use ONLY the SimCLR-pretrained "
                     "BrainIAC.ckpt from the BrainIAC repo's Dropbox, not the task checkpoints (those have seen other datasets).")
print("checkpoint:", CKPT_PATH or "NONE (self-test, random weights)")

# ------------------------------------ data index -------------------------------------
ALIAS = {"normal": "normal", "control": "normal", "controls": "normal", "cn": "normal", "healthy": "normal", "nc": "normal",
         "mci": "mci", "alzheimers": "alzheimers", "alzheimer": "alzheimers", "ad": "alzheimers", "dementia": "alzheimers",
         "stroke": "stroke"}
key = lambda s: re.sub(r"[^a-z]", "", str(s).lower())

def discover():
    csv = os.path.join(DATA_ROOT, "labels.csv") if DATA_ROOT else ""
    if CLASS_DIRS and not (csv and os.path.exists(csv)):
        rows = []
        for c, d in CLASS_DIRS.items():
            fs = sorted(glob.glob(f"{d}/**/*.nii*", recursive=True)); assert fs, f"no .nii/.nii.gz files under {d} (class {c})"
            rows += [(f, c) for f in fs]
        df = pd.DataFrame(rows, columns=["path", "label"])
    elif os.path.exists(csv):
        df = pd.read_csv(csv)
        assert {"path", "label"} <= set(df.columns), "labels.csv needs the columns: path,label (optional: subject, age)"
        df["path"] = [p if os.path.isabs(p) else os.path.join(DATA_ROOT, p) for p in df.path]
    else:
        rows = []
        for d in sorted(os.listdir(DATA_ROOT)):
            k = ALIAS.get(key(d))
            if k and os.path.isdir(f"{DATA_ROOT}/{d}"):
                rows += [(f, k) for f in sorted(glob.glob(f"{DATA_ROOT}/{d}/**/*.nii*", recursive=True))]
        df = pd.DataFrame(rows, columns=["path", "label"])
    df["label"] = [ALIAS.get(key(l), str(l).lower()) for l in df.label]
    bad = sorted(set(df.label) - set(CLASSES)); assert not bad, f"unknown labels {bad}; expected {CLASSES}"
    miss = ~df.path.map(os.path.exists); assert not miss.any(), f"{miss.sum()} files missing, e.g. {df.path[miss].iloc[0]}"
    def subj(p):
        b = os.path.basename(p)
        for rx in ([SUBJECT_REGEX] if isinstance(SUBJECT_REGEX, str) else (SUBJECT_REGEX or [])):
            m = re.search(rx, b)
            if m: return m.group(1)
        return re.sub(r"\.nii(\.gz)?$", "", b)
    if "subject" not in df.columns: df["subject"] = [subj(p) for p in df.path]
    df["y"] = df.label.map({c: i for i, c in enumerate(CLASSES)})
    return df.reset_index(drop=True)

df = discover(); N = len(df)
Y = df.y.values; G = pd.factorize(df.subject)[0]
counts = df.label.value_counts().reindex(CLASSES).fillna(0).astype(int)
print("scans per class:", counts.to_dict(), "| subjects:", df.subject.nunique())
if df.subject.nunique() == N: print("NOTE: every scan has its own subject id. If one person has several scans (e.g. several MRI sequences), set SUBJECT_REGEX or a 'subject' column.")
gpc = df.groupby("label").subject.nunique().reindex(CLASSES)
assert (gpc >= FOLDS).all(), f"every class needs >= {FOLDS} subjects for {FOLDS}-fold CV; have {gpc.to_dict()}. Lower FOLDS."
if counts.max() / counts.min() > 3: print("WARNING: strong class imbalance; class weights are used, and read balanced accuracy / macro-F1, not plain accuracy.")

def load_scan(path):                      # identical to jetson_infer.py: trilinear resize to 96^3, z-score over non-zero voxels
    a = np.nan_to_num(nib.load(path).get_fdata(dtype=np.float32))
    if a.ndim == 4: a = a[..., 0]
    x = F.interpolate(torch.from_numpy(a)[None, None], size=(96, 96, 96), mode="trilinear", align_corners=False)
    nz = x != 0
    if nz.sum() > 1:
        m, s = x[nz].mean(), x[nz].std(unbiased=False)
        x = torch.where(nz, (x - m) / (s if s > 0 else 1.0), x)
    return x[0].contiguous(), a            # [1,96,96,96] float32, raw array

def scan_stats(a, x):                      # simple image descriptors, used ONLY for the dataset-shortcut check below
    v = a[a != 0]; z = x[x != 0].numpy() if (x != 0).any() else np.zeros(1)
    if v.size < 10: v = np.zeros(10)
    q = np.percentile(z, [5, 25, 50, 75, 95]); sk = float(((z - z.mean()) ** 3).mean() / (z.std() ** 3 + 1e-8))
    return [float(v.mean()), float(v.std()), *np.percentile(v, [5, 50, 95]).tolist(), float((a != 0).mean()), *a.shape[:3], *q.tolist(), sk]

IMG, STATS = [], []
t0 = time.time()
for p_ in df.path:
    x_, a_ = load_scan(p_); IMG.append(x_); STATS.append(scan_stats(a_, x_))
STATS = np.array(STATS)
print(f"loaded {N} scans into RAM in {time.time()-t0:.0f}s ({N*96**3*4/1e9:.1f} GB)")
shapes = {tuple(nib.load(p).shape[:3]) for p in df.path[:min(N, 50)]}
if len(shapes) > 1: print("WARNING: different input shapes", shapes, "- scans should all be registered to the same atlas grid.")

# ------------------------------------ model ------------------------------------------
class BrainIACClassifier(nn.Module):
    """BrainIAC ViT-B backbone + linear head. Feature = first output token (as in the BrainIAC repo)."""
    def __init__(self, n_classes, ckpt, layers=12):
        super().__init__()
        self.vit = ViT(in_channels=1, img_size=(96, 96, 96), patch_size=(16, 16, 16), hidden_size=768, mlp_dim=3072,
                       num_layers=layers, num_heads=12, save_attn=False)
        if ckpt:
            sd = torch.load(ckpt, map_location="cpu", weights_only=False); sd = sd.get("state_dict", sd)
            sd = {k[len("backbone."):]: v for k, v in sd.items() if k.startswith("backbone.")}
            missing, unexpected = self.vit.load_state_dict(sd, strict=False)
            assert not unexpected, f"unexpected keys: {unexpected[:5]}"
            assert all("norm_cross_attn" in k for k in missing), f"missing keys: {[k for k in missing if 'norm_cross_attn' not in k][:5]}"
            print(f"  backbone loaded: {len(sd)} tensors ({len(missing)} unused cross-attention norms left at default)") if not getattr(BrainIACClassifier, "_p", 0) else None
            BrainIACClassifier._p = 1
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(768, n_classes))
    def forward(self, x):
        return self.head(self.vit(x)[0][:, 0])      # logits [B, n_classes]

def build(mode):
    m = BrainIACClassifier(C, CKPT_PATH, LAYERS)
    if mode == "linear":
        for p in m.vit.parameters(): p.requires_grad = False
    return m.to(DEV)

def make_opt(model, mode, epochs):
    groups = [{"params": model.head.parameters(), "lr": LR_HEAD}]
    if mode == "full": groups.append({"params": model.vit.parameters(), "lr": LR_BACKBONE})
    opt = torch.optim.AdamW(groups, weight_decay=0.05)
    warm = 3
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e + 1) / warm if e < warm else 0.5 * (1 + math.cos(math.pi * (e - warm) / max(1, epochs - warm))))
    return opt, sched, torch.amp.GradScaler("cuda", enabled=(DEV == "cuda"))

TF = Compose([RandFlip(prob=0.5, spatial_axis=0),
              RandAffine(prob=0.7, rotate_range=(0.1,) * 3, scale_range=(0.08,) * 3, translate_range=(4,) * 3, padding_mode="zeros", device=DEV),
              RandScaleIntensity(factors=0.1, prob=0.5), RandShiftIntensity(offsets=0.1, prob=0.5), RandGaussianNoise(prob=0.2, std=0.05)])
def aug(x):
    y = TF(x); return y.as_tensor() if hasattr(y, "as_tensor") else y

def train_epoch(model, idx, opt, scaler, wts):
    model.train(); idx = np.random.permutation(idx); tot = n = 0
    for i in range(0, len(idx), BATCH):
        b = idx[i:i + BATCH]
        if len(b) < 2: continue
        x = torch.stack([aug(IMG[j].to(DEV)) for j in b]); y = torch.as_tensor(Y[b], device=DEV)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEV == "cuda")):
            loss = F.cross_entropy(model(x).float(), y, weight=wts, label_smoothing=0.05)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
        tot += loss.item() * len(b); n += len(b)
    return tot / max(n, 1)

@torch.no_grad()
def predict(model, idx):
    model.eval(); out = []
    for i in range(0, len(idx), 8):
        x = torch.stack([IMG[j] for j in idx[i:i + 8]]).to(DEV)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEV == "cuda")):
            out.append(F.softmax(model(x).float(), 1).cpu().numpy())
    return np.concatenate(out).astype(np.float64)

def class_weights(idx):
    n = np.bincount(Y[idx], minlength=C).astype(float); w = len(idx) / (C * np.maximum(n, 1))
    return torch.tensor(w, dtype=torch.float32, device=DEV)

# ------------------------------------ metrics ----------------------------------------
def macro_auc(y, P):
    ok = [c for c in range(C) if (y == c).any() and (y != c).any()]
    return float(np.mean([roc_auc_score(y == c, P[:, c]) for c in ok]))
def metrics(y, P):
    pr = P.argmax(1)
    return dict(acc=accuracy_score(y, pr), bal_acc=balanced_accuracy_score(y, pr),
                macro_f1=f1_score(y, pr, labels=list(range(C)), average="macro", zero_division=0), auc=macro_auc(y, P))
def boot_ci(y, P, fn, n=1000, seed=0):
    r = np.random.RandomState(seed); v = []
    for _ in range(n):
        i = r.randint(0, len(y), len(y))
        try: v.append(fn(y[i], P[i]))
        except ValueError: pass
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

# ------------------------------------ cross-validation -------------------------------
SPLITS = list(StratifiedGroupKFold(n_splits=FOLDS, shuffle=True, random_state=SEED).split(np.zeros(N), Y, G))

# ---- dataset-shortcut check: can trivial image statistics (no brain anatomy) already separate the classes? ----
from sklearn.ensemble import RandomForestClassifier
Ps = np.zeros((N, C))
for tr, te in SPLITS:
    rf = RandomForestClassifier(300, random_state=SEED, class_weight="balanced").fit(STATS[tr], Y[tr]); Ps[np.ix_(te, rf.classes_)] = rf.predict_proba(STATS[te])
SHORT = {CLASSES[c]: roc_auc_score(Y == c, Ps[:, c]) for c in range(C)}
print("\nSHORTCUT CHECK (random forest on intensity/shape statistics only, out-of-fold one-vs-rest AUC):", {k: round(v, 3) for k, v in SHORT.items()})
for k, v in SHORT.items():
    if v > 0.9: print(f"  !! '{k}' is separable from the rest with {v:.2f} AUC using trivial image statistics. It probably comes from a different dataset/scanner/sequence, "
                      f"so a high '{k}' score from the model would NOT show that it recognises the disease. Quote the {WITHIN_SOURCE} results for the disease claims.")

def run_cv(mode):
    E = EPOCHS[mode]; oof = np.zeros((N, C)); fold_of = np.zeros(N, int); hist, fold_rows, best_eps = [], [], []
    for f, (tr, te) in enumerate(SPLITS):
        t0 = time.time()
        ns = int(np.clip(min(len(set(G[tr][Y[tr] == c])) for c in range(C)), 2, 5))     # inner split: validation scans pick the epoch
        a, b = next(StratifiedGroupKFold(n_splits=ns, shuffle=True, random_state=SEED + f).split(tr, Y[tr], G[tr]))
        trn, val = tr[a], tr[b]
        model = build(mode); opt, sched, scaler = make_opt(model, mode, E); wts = class_weights(trn)
        best = (-1.0, -1e9, None, 0)
        for ep in range(E):
            tl = train_epoch(model, trn, opt, scaler, wts); sched.step()
            Pv = predict(model, val); va = macro_auc(Y[val], Pv); vl = float(-np.log(np.clip(Pv[np.arange(len(val)), Y[val]], 1e-7, 1)).mean())
            hist.append(dict(mode=mode, fold=f, epoch=ep + 1, train_loss=tl, val_loss=vl, val_auc=va))
            if (va, -vl) > (best[0], best[1]): best = (va, -vl, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ep + 1)
        model.load_state_dict(best[2]); Pt = predict(model, te); oof[te] = Pt; fold_of[te] = f; best_eps.append(best[3])
        m = metrics(Y[te], Pt); fold_rows.append(dict(mode=mode, fold=f, n_test=len(te), best_epoch=best[3], val_auc=best[0], **m))
        print(f"[{mode}] fold {f}: test macro-AUC {m['auc']:.3f} | bal-acc {m['bal_acc']:.3f} | acc {m['acc']:.3f} | best epoch {best[3]}/{E} | {(time.time()-t0)/60:.1f} min", flush=True)
        del model, opt, best; torch.cuda.empty_cache()
    pooled = metrics(Y, oof)
    pooled["auc_ci"] = boot_ci(Y, oof, macro_auc); pooled["bal_acc_ci"] = boot_ci(Y, oof, lambda y, P: balanced_accuracy_score(y, P.argmax(1)))
    return dict(oof=oof, fold_of=fold_of, hist=pd.DataFrame(hist), folds=pd.DataFrame(fold_rows), best_eps=best_eps, pooled=pooled)

CV = {}
for mode in MODES:
    print(f"\n===== {mode} | {FOLDS}-fold CV | {N} scans | {EPOCHS[mode]} epochs/fold =====")
    CV[mode] = run_cv(mode)
BEST = max(MODES, key=lambda m: CV[m]["pooled"]["auc"]); R = CV[BEST]; oof = R["oof"]; pred = oof.argmax(1)
print(f"\nselected mode: {BEST}")

# age-only baseline (confound check) if the labels file has an 'age' column
age_auc = None
if "age" in df.columns and df.age.notna().all():
    Pa = np.zeros((N, C))
    for tr, te in SPLITS:
        clf = LogisticRegression(max_iter=1000).fit(df.age.values[tr, None], Y[tr]); Pa[np.ix_(te, clf.classes_)] = clf.predict_proba(df.age.values[te, None])
    age_auc = macro_auc(Y, Pa); print(f"age-only baseline macro-AUC: {age_auc:.3f}  (the model should clearly beat this)")

# ------------------------------------ reports ----------------------------------------
P_ = R["pooled"]
prec, rec, f1, sup = precision_recall_fscore_support(Y, pred, labels=list(range(C)), zero_division=0)
per_auc = [roc_auc_score(Y == c, oof[:, c]) for c in range(C)]
rep = pd.DataFrame(dict(precision=prec, recall=rec, f1=f1, auc_ovr=per_auc, support=sup), index=CLASSES).round(3)
cm = confusion_matrix(Y, pred, labels=list(range(C)))
print("\n=== POOLED OUT-OF-FOLD RESULTS (each scan predicted by a model that never saw its subject) ===")
print(f"macro AUC {P_['auc']:.3f} (95% CI {P_['auc_ci'][0]:.2f}-{P_['auc_ci'][1]:.2f}) | balanced acc {P_['bal_acc']:.3f} "
      f"(95% CI {P_['bal_acc_ci'][0]:.2f}-{P_['bal_acc_ci'][1]:.2f}) | accuracy {P_['acc']:.3f} | macro-F1 {P_['macro_f1']:.3f}")
print(rep.to_string()); print("confusion matrix (rows=true, cols=predicted):\n", pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_string())
fm = R["folds"]; print("\nper-fold:\n", fm.round(3).to_string(index=False))

WS = [CLASSES.index(c) for c in WITHIN_SOURCE if c in CLASSES]; WSD = None
if len(WS) >= 2:
    keep = np.isin(Y, WS); Pw = oof[keep][:, WS]; Pw = Pw / np.maximum(Pw.sum(1, keepdims=True), 1e-9); yw = np.array([WS.index(v) for v in Y[keep]])
    pw = Pw.argmax(1); k_ = len(WS)
    aucs = [roc_auc_score(yw == i, Pw[:, i]) for i in range(k_)]
    WSD = dict(classes=[CLASSES[i] for i in WS], n=int(keep.sum()), macro_auc=float(np.mean(aucs)), per_class_auc=[float(a) for a in aucs],
               acc=float(accuracy_score(yw, pw)), bal_acc=float(balanced_accuracy_score(yw, pw)), macro_f1=float(f1_score(yw, pw, average="macro", zero_division=0)))
    print(f"\n=== WITHIN-SOURCE RESULT ({' / '.join(WSD['classes'])} only; probabilities renormalised over these classes; n={WSD['n']}) ===")
    print(f"macro AUC {WSD['macro_auc']:.3f} | balanced acc {WSD['bal_acc']:.3f} | accuracy {WSD['acc']:.3f} | macro-F1 {WSD['macro_f1']:.3f} | per-class AUC "
          + ", ".join(f"{c} {a:.2f}" for c, a in zip(WSD["classes"], aucs)))
    cmw = confusion_matrix(yw, pw, labels=list(range(k_))); print(pd.DataFrame(cmw, index=WSD["classes"], columns=WSD["classes"]).to_string())
    json.dump(WSD, open(f"{OUT}/within_source_metrics.json", "w"), indent=2)

fm_all = pd.concat([CV[m]["folds"] for m in MODES]); fm_all.to_csv(f"{OUT}/cv_fold_metrics.csv", index=False)
pd.concat([CV[m]["hist"] for m in MODES]).to_csv(f"{OUT}/training_history.csv", index=False)
rep.to_csv(f"{OUT}/classification_report.csv")
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(f"{OUT}/confusion_matrix.csv")
o = df[["path", "label", "subject"]].copy(); o["fold"] = R["fold_of"]; o["pred"] = [CLASSES[i] for i in pred]
for c in range(C): o[f"p_{CLASSES[c]}"] = oof[:, c]
o.to_csv(f"{OUT}/oof_predictions.csv", index=False)

# ------------------------------------ graphs -----------------------------------------
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]          # fixed class colours: normal, mci, alzheimers, stroke
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "grid.linewidth": .6,
                     "font.size": 10, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK, "xtick.color": MUTED,
                     "ytick.color": MUTED, "figure.facecolor": "white", "axes.facecolor": "white", "legend.frameon": False})
NICE = {"normal": "Normal", "mci": "MCI", "alzheimers": "Alzheimer's", "stroke": "Stroke"}
lab = [NICE.get(c, c) for c in CLASSES]
def save(fig, name): fig.tight_layout(); fig.savefig(f"{OUT}/plots/{name}.png", dpi=150); plt.close(fig)

fig, ax = plt.subplots(figsize=(5.5, 3.4))                                              # 1 class balance
bars = ax.bar(lab, counts.values, color=COL[:C], width=.6)
for b_, v in zip(bars, counts.values): ax.text(b_.get_x() + b_.get_width() / 2, v, str(v), ha="center", va="bottom", color=INK)
ax.set_ylabel("scans"); ax.set_title("Dataset: scans per class", loc="left"); ax.grid(axis="x", visible=False); save(fig, "1_class_balance")

for mode in MODES:                                                                      # 2 training curves
    h = CV[mode]["hist"]; fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
    for f_, g in h.groupby("fold"):
        axs[0].plot(g.epoch, g.train_loss, color=COL[0], alpha=.18, lw=1); axs[0].plot(g.epoch, g.val_loss, color=COL[1], alpha=.18, lw=1)
        axs[1].plot(g.epoch, g.val_auc, color=COL[0], alpha=.18, lw=1)
    mh = h.groupby("epoch")[["train_loss", "val_loss", "val_auc"]].mean()
    axs[0].plot(mh.index, mh.train_loss, color=COL[0], lw=2, label="train loss"); axs[0].plot(mh.index, mh.val_loss, color=COL[1], lw=2, label="validation loss")
    axs[1].plot(mh.index, mh.val_auc, color=COL[0], lw=2, label="validation macro-AUC")
    axs[0].set_title(f"Loss ({mode}; thin = folds, bold = mean)", loc="left"); axs[1].set_title("Validation macro-AUC", loc="left")
    for a_ in axs: a_.set_xlabel("epoch")
    axs[0].legend(); axs[1].set_ylim(0, 1.02); save(fig, f"2_training_curves_{mode}")

fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))                                        # 3 confusion matrices
for a_, (M, ttl, fmt) in zip(axs, [(cm, "Counts", "d"), (cm / np.maximum(cm.sum(1, keepdims=True), 1), "Row-normalised (recall)", ".2f")]):
    a_.imshow(M, cmap="Blues", vmin=0, vmax=M.max()); a_.grid(False)
    for i in range(C):
        for j in range(C): a_.text(j, i, format(M[i, j], fmt), ha="center", va="center", color="white" if M[i, j] > M.max() * .55 else INK)
    a_.set_xticks(range(C)); a_.set_xticklabels(lab, rotation=25); a_.set_yticks(range(C)); a_.set_yticklabels(lab)
    a_.set_xlabel("predicted"); a_.set_ylabel("true"); a_.set_title(ttl, loc="left")
save(fig, "3_confusion_matrix")

fig, ax = plt.subplots(figsize=(5.8, 5))                                                # 4 ROC one-vs-rest
for c in range(C):
    fpr, tpr, _ = roc_curve(Y == c, oof[:, c]); ax.plot(fpr, tpr, color=COL[c], lw=2, label=f"{lab[c]} (AUC {per_auc[c]:.2f})")
ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--"); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title(f"ROC, one-vs-rest, out-of-fold (macro-AUC {P_['auc']:.2f})", loc="left"); ax.legend(loc="lower right"); ax.set_aspect("equal"); save(fig, "4_roc_curves")

fig, ax = plt.subplots(figsize=(7.5, 3.8)); w = .26; xs = np.arange(C)                  # 5 per-class metrics
for k, (name, v, cc) in enumerate([("precision", prec, COL[0]), ("recall", rec, COL[1]), ("F1", f1, COL[2])]):
    bb = ax.bar(xs + (k - 1) * w, v, w * .92, color=cc, label=name)
    for b_, val in zip(bb, v): ax.text(b_.get_x() + b_.get_width() / 2, val + .01, f"{val:.2f}", ha="center", fontsize=8, color=MUTED)
ax.set_xticks(xs); ax.set_xticklabels(lab); ax.set_ylim(0, 1.12); ax.set_title("Per-class precision / recall / F1 (out-of-fold)", loc="left")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(.5, -.12)); ax.grid(axis="x", visible=False); save(fig, "5_per_class_metrics")

fig, ax = plt.subplots(figsize=(6.5, 3.6))                                              # 6 fold stability
for k, (met, cc) in enumerate([("auc", COL[0]), ("bal_acc", COL[1]), ("macro_f1", COL[2])]):
    ax.plot(fm.fold, fm[met], "o-", color=cc, lw=1.6, ms=6, label={"auc": "macro-AUC", "bal_acc": "balanced acc", "macro_f1": "macro-F1"}[met])
if age_auc: ax.axhline(age_auc, color=MUTED, ls="--", lw=1, label=f"age-only AUC {age_auc:.2f}")
ax.axhline(1 / C, color=MUTED, ls=":", lw=1, label="chance accuracy"); ax.set_xticks(fm.fold); ax.set_xlabel("fold"); ax.set_ylim(0, 1.05)
ax.set_title("Fold-to-fold stability", loc="left"); ax.legend(ncol=2, fontsize=8); save(fig, "6_fold_stability")

if len(MODES) > 1:                                                                      # 7 mode comparison
    fig, ax = plt.subplots(figsize=(5.5, 3.6)); xs = np.arange(len(MODES))
    for k, (met, cc) in enumerate([("auc", COL[0]), ("bal_acc", COL[1])]):
        ax.bar(xs + (k - .5) * .34, [CV[m]["pooled"][met] for m in MODES], .32, color=cc, label={"auc": "macro-AUC", "bal_acc": "balanced acc"}[met])
    ax.set_xticks(xs); ax.set_xticklabels(MODES); ax.set_ylim(0, 1.05); ax.legend(); ax.set_title("Linear-probe vs full fine-tune", loc="left"); save(fig, "7_mode_comparison")
print("graphs saved to", f"{OUT}/plots:", sorted(os.listdir(f"{OUT}/plots")))

# ------------------------------------ final model on ALL scans ------------------------
mode = BEST; E = EPOCHS[mode]; ep = int(np.clip(np.median(R["best_eps"]), min(10, E), E))
print(f"\n===== final model: {mode}, all {N} scans, {ep} epochs =====")
final = build(mode); opt, sched, scaler = make_opt(final, mode, ep); allx = np.arange(N); wts = class_weights(allx)
for e in range(ep):
    l = train_epoch(final, allx, opt, scaler, wts); sched.step()
    if e == 0 or (e + 1) % 10 == 0 or e == ep - 1: print(f"   epoch {e+1}/{ep}  train loss {l:.3f}", flush=True)
torch.save(final.state_dict(), f"{OUT}/weights/brainiac_{C}class_{mode}.pt")
json.dump(dict(classes=CLASSES, mode=mode, epochs=ep, layers=LAYERS, head="Dropout(0.2)+Linear(768,C)"), open(f"{OUT}/weights/brainiac_{C}class_config.json", "w"))

# ------------------------------------ ONNX + Jetson package --------------------------
D = f"{OUT}/deploy"; shutil.rmtree(D, ignore_errors=True); os.makedirs(D)
net = copy.deepcopy(final).eval().cpu().float(); onnx_path = f"{D}/brainiac_multiclass.onnx"
kw = dict(input_names=["mri"], output_names=["logits"], opset_version=17, do_constant_folding=True)
try: torch.onnx.export(net, torch.randn(1, 1, 96, 96, 96), onnx_path, dynamo=False, **kw)
except TypeError: torch.onnx.export(net, torch.randn(1, 1, 96, 96, 96), onnx_path, **kw)
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"]); diffs, agree = [], []
for i in range(min(4, N)):
    x = IMG[i].unsqueeze(0)
    with torch.no_grad(): a = net(x).numpy()
    b = sess.run(None, {"mri": x.numpy()})[0]; diffs.append(float(np.abs(a - b).max())); agree.append(int(a.argmax() == b.argmax()))
print(f"ONNX vs PyTorch: max |logit diff| {max(diffs):.5f} (want ~1e-3 or lower) | same predicted class on {sum(agree)}/{len(agree)} scans")

sid = 0; x = np.ascontiguousarray(IMG[sid].unsqueeze(0).numpy().astype(np.float32)); x.tofile(f"{D}/sample_input.bin")
with torch.no_grad(): lg = net(torch.from_numpy(x)).numpy()[0]
pr_ = np.exp(lg - lg.max()); pr_ /= pr_.sum()
json.dump(dict(scan=os.path.basename(df.path[sid]), true_label=df.label[sid], logits=lg.tolist(), probs=pr_.tolist(), pred=CLASSES[int(pr_.argmax())]), open(f"{D}/expected.json", "w"), indent=2)
if SAMPLE_SCAN_IN_PACKAGE: shutil.copy(df.path[sid], f"{D}/sample_scan.nii.gz")
atlas = glob.glob("/kaggle/working/BrainIAC/src/preprocessing/atlases/temp_head.nii.gz") + glob.glob("**/temp_head.nii.gz", recursive=True)
if atlas: shutil.copy(atlas[0], f"{D}/temp_head.nii.gz")
spec = dict(task=f"{C}-class brain MRI classifier: " + ", ".join(CLASSES), input_name="mri", input_shape=[1, 1, 96, 96, 96],
            output_name="logits", output_shape=[1, C], classes=CLASSES, output="logits; probabilities = softmax(logits); prediction = argmax",
            resize="trilinear (torch interpolate, align_corners=False) from the stored grid to (96,96,96)",
            normalize="z-score over non-zero voxels only (zeros stay zero), population std",
            upstream="N4 bias correction -> resample 1 mm -> rigid registration to temp_head.nii.gz -> HD-BET skull strip (NIfTI)",
            training=dict(n_scans=N, per_class=counts.to_dict(), mode=mode, epochs=ep, base="BrainIAC SimCLR ViT-B"),
            cv=dict(folds=FOLDS, macro_auc=P_["auc"], macro_auc_ci=P_["auc_ci"], balanced_accuracy=P_["bal_acc"], accuracy=P_["acc"], macro_f1=P_["macro_f1"],
                    per_class=rep.to_dict("index"), age_only_macro_auc=age_auc, within_source=WSD, shortcut_check_auc=SHORT, max_onnx_logit_diff=max(diffs)))
json.dump(spec, open(f"{D}/spec.json", "w"), indent=2)

open(f"{D}/jetson_infer.py", "w").write(textwrap.dedent(r'''
    import json, time, argparse
    import numpy as np, torch, torch.nn.functional as F, nibabel as nib, tensorrt as trt

    def preprocess(path):                          # skull-stripped, atlas-registered NIfTI -> model input
        a = np.nan_to_num(nib.load(path).get_fdata().astype(np.float32))
        x = F.interpolate(torch.from_numpy(a)[None, None], size=(96, 96, 96), mode="trilinear", align_corners=False)
        nz = x != 0; m, s = x[nz].mean(), x[nz].std(unbiased=False)
        return torch.where(nz, (x - m) / (s if s > 0 else 1.0), x).contiguous()

    class Engine:
        def __init__(self, path, n_classes):
            self.engine = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(open(path, "rb").read())
            self.ctx = self.engine.create_execution_context()
            self.out = torch.empty((1, n_classes), dtype=torch.float32, device="cuda")
        def __call__(self, x):
            x = x.to("cuda").float().contiguous()
            self.ctx.set_tensor_address("mri", x.data_ptr()); self.ctx.set_tensor_address("logits", self.out.data_ptr())
            ok = self.ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream); torch.cuda.synchronize()
            assert ok, "TensorRT execution failed"
            return self.out[0].cpu().numpy()

    if __name__ == "__main__":
        ap = argparse.ArgumentParser()
        ap.add_argument("--engine", required=True); ap.add_argument("--scan", required=True); ap.add_argument("--spec", default="spec.json")
        ap.add_argument("--expected"); ap.add_argument("--iters", type=int, default=50)
        a = ap.parse_args(); spec = json.load(open(a.spec)); cls = spec["classes"]
        eng = Engine(a.engine, len(cls)); x = preprocess(a.scan); eng(x)                      # warm-up
        t = []
        for _ in range(a.iters):
            t0 = time.perf_counter(); lg = eng(x); t.append((time.perf_counter() - t0) * 1000)
        p = np.exp(lg - lg.max()); p /= p.sum()
        print("prediction:", cls[int(p.argmax())], "| " + "  ".join(f"{c} {v:.3f}" for c, v in zip(cls, p)))
        print(f"engine latency over {a.iters} runs: mean {np.mean(t):.1f} ms | median {np.median(t):.1f} ms | p95 {np.percentile(t, 95):.1f} ms")
        if a.expected:
            e = json.load(open(a.expected)); print("expected:", e["pred"], "| max |logit diff| vs PyTorch:", float(np.abs(np.array(e["logits"]) - lg).max()))
'''))

cvtxt = (f"- {FOLDS}-fold subject-grouped cross-validation, pooled out-of-fold: macro-AUC {P_['auc']:.2f} (95% CI {P_['auc_ci'][0]:.2f}-{P_['auc_ci'][1]:.2f}), "
         f"balanced accuracy {P_['bal_acc']:.2f}, accuracy {P_['acc']:.2f}, macro-F1 {P_['macro_f1']:.2f}.")
open(f"{D}/README_FOR_JETSON.md", "w").write("\n".join([
    f"# BrainIAC {C}-class brain MRI classifier ({', '.join(CLASSES)}), research prototype", "",
    "NOT for clinical use. BrainIAC is licensed for non-commercial academic research; the data licences of the training sets apply.", "",
    f"Fine-tuned from BrainIAC (3D ViT-B) on {N} scans ({counts.to_dict()}). Final model trained on all scans, mode={mode}, {ep} epochs.",
    cvtxt, (f"- Within-source ({' / '.join(WSD['classes'])}): macro-AUC {WSD['macro_auc']:.2f}, balanced accuracy {WSD['bal_acc']:.2f}, macro-F1 {WSD['macro_f1']:.2f}. This is the fair disease number." if WSD else ""),
    f"- Dataset-shortcut check (trivial image statistics, one-vs-rest AUC): { {k: round(v, 2) for k, v in SHORT.items()} }. Classes near 1.0 come from a different dataset/sequence, so their score reflects the source, not the disease.",
    "- The delivered model saw every scan, so use the CV numbers above, never a score on training scans.", "",
    "## Files", "- brainiac_multiclass.onnx  input `mri` [1,1,96,96,96] float32 -> output `logits` [1,%d]; probabilities = softmax(logits); classes in order: %s" % (C, CLASSES),
    "- spec.json (full contract + metrics), sample_input.bin + expected.json (reference output), sample_scan.nii.gz (if included), jetson_infer.py, temp_head.nii.gz (atlas, if found)", "",
    "## Build and test on the Jetson (build the engine ON the device)",
    "    cat /etc/nv_tegra_release",
    "    python3 -c \"import tensorrt, torch; print(tensorrt.__version__, torch.__version__, torch.cuda.is_available())\"",
    "    sudo nvpmodel -m 0 && sudo jetson_clocks",
    "    /usr/src/tensorrt/bin/trtexec --onnx=brainiac_multiclass.onnx --saveEngine=brainiac_fp16.engine --fp16",
    "    /usr/src/tensorrt/bin/trtexec --loadEngine=brainiac_fp16.engine --loadInputs=mri:sample_input.bin --dumpOutput --iterations=50",
    "    python3 jetson_infer.py --engine brainiac_fp16.engine --scan sample_scan.nii.gz --spec spec.json --expected expected.json",
    "In another terminal during the runs: `tegrastats` (RAM, GPU load, power) or `jtop`. Report mean/p95 latency, peak RAM, power mode, JetPack + TensorRT versions.",
    "FP16 logits should be within ~0.01-0.1 of expected.json and give the same class. If not, retry without --fp16. If the ONNX parse fails on old TensorRT, ask for an opset-16 export.", "",
    "## Before the model (NOT in the engine)",
    "Raw T1 scans need: N4 bias correction -> 1 mm resample -> rigid registration to temp_head.nii.gz -> HD-BET skull strip, giving a NIfTI on the same grid as the training data. "
    "jetson_infer.py then resizes to 96^3 and z-scores. Registration took ~3-4 min per scan on a Kaggle CPU and is untimed on the Orin."]))

shutil.make_archive(f"{OUT}/jetson_package_{C}class", "zip", D)
shutil.make_archive(f"{OUT}/weights_{C}class", "zip", f"{OUT}/weights")
stage = f"{OUT}/_bundle"; shutil.rmtree(stage, ignore_errors=True); shutil.copytree(f"{OUT}/plots", f"{stage}/plots")
for f in glob.glob(f"{OUT}/*.csv"): shutil.copy(f, stage)
shutil.copy(f"{D}/spec.json", f"{stage}/metrics_and_spec.json"); shutil.make_archive(f"{OUT}/results_bundle_{C}class", "zip", stage); shutil.rmtree(stage)
print("\n=== DONE. Download these three zips from", OUT, "===")
for z in (f"jetson_package_{C}class", f"weights_{C}class", f"results_bundle_{C}class"):
    print(f"  {z}.zip  {os.path.getsize(f'{OUT}/{z}.zip')/1e6:.0f} MB")
try:
    from IPython.display import FileLink, display
    for z in (f"jetson_package_{C}class", f"weights_{C}class", f"results_bundle_{C}class"):
        rel = os.path.relpath(f"{OUT}/{z}.zip", os.getcwd()); display(FileLink(rel))
except Exception: pass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.3 MB/s eta 0:00:00
device: cuda (Tesla T4)
checkpoint: /kaggle/working/ckpt/BrainIAC.ckpt
scans per class: {'normal': 15, 'mci': 15, 'alzheimers': 15, 'stroke': 15} | subjects: 60
NOTE: every scan has its own subject id. If one person has several scans (e.g. several MRI sequences), set SUBJECT_REGEX or a 'subject' column.
loaded 60 scans into RAM in 14s (0.2 GB)

SHORTCUT CHECK (random forest on intensity/shape statistics only, out-of-fold one-vs-rest AUC): {'normal': np.float64(0.997), 'mci': np.float64(0.865), 'alzheimers': np.float64(0.877), 'stroke': np.float64(0.996)}
  !! 'normal' is separable from the rest with 1.00 AUC using trivial image statistics. It probably comes from a different dataset/scanner/sequence, so a high 'normal' score from the model would NOT show that it recognises the disease. Quote the ['normal', 'mci', 'alzheimers'] result

/kaggle/working/results_mc/jetson_package_4class.zip

/kaggle/working/results_mc/weights_4class.zip

/kaggle/working/results_mc/results_bundle_4class.zip

In [12]:
# Standalone metrics + plots from the saved out-of-fold predictions. Works after a session restart (needs only the CSV).
import warnings; warnings.filterwarnings("ignore")
import os, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
                             precision_recall_curve, average_precision_score, confusion_matrix, matthews_corrcoef, cohen_kappa_score, log_loss)

CSV    = os.environ.get("OOF_CSV", "/kaggle/working/results_mc/oof_predictions.csv")
OUT    = os.environ.get("PLOT_OUT", "/kaggle/working/results_mc/metrics_plots")
WITHIN = ["normal", "mci", "alzheimers"]                       # classes from one dataset: reported separately
os.makedirs(OUT, exist_ok=True)

d = pd.read_csv(CSV); CL = [c[2:] for c in d.columns if c.startswith("p_")]; K = len(CL)
P = d[[f"p_{c}" for c in CL]].values; y = d.label.map({c: i for i, c in enumerate(CL)}).values; pred = P.argmax(1)
NICE = {"normal": "Normal", "mci": "MCI", "alzheimers": "Alzheimer's", "stroke": "Stroke"}; lab = [NICE.get(c, c) for c in CL]
COL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"][:K]; INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "font.size": 10,
                     "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False})
def fin(fig, name): fig.tight_layout(); fig.savefig(f"{OUT}/{name}.png", dpi=150); plt.show()

def macro_auc(y, P): return float(np.mean([roc_auc_score(y == c, P[:, c]) for c in range(P.shape[1]) if (y == c).any() and (y != c).any()]))
def boot(fn, y, P, n=1000):
    r = np.random.RandomState(0); v = []
    for _ in range(n):
        i = r.randint(0, len(y), len(y))
        try: v.append(fn(y[i], P[i]))
        except ValueError: pass
    return np.percentile(v, [2.5, 97.5])

# ---------------- overall metrics ----------------
ci_auc = boot(macro_auc, y, P); ci_bal = boot(lambda y, P: balanced_accuracy_score(y, P.argmax(1)), y, P)
overall = {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
           "macro_precision": precision_score(y, pred, average="macro", zero_division=0), "macro_recall": recall_score(y, pred, average="macro", zero_division=0),
           "macro_f1": f1_score(y, pred, average="macro", zero_division=0), "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
           "macro_auc_ovr": macro_auc(y, P), "macro_avg_precision": float(np.mean([average_precision_score(y == c, P[:, c]) for c in range(K)])),
           "mcc": matthews_corrcoef(y, pred), "cohen_kappa": cohen_kappa_score(y, pred), "log_loss": log_loss(y, np.clip(P, 1e-7, 1), labels=list(range(K)))}
print(f"n = {len(y)} scans | classes: {CL}\n")
print(pd.Series(overall).round(3).to_string())
print(f"\nmacro-AUC 95% CI {ci_auc[0]:.2f}-{ci_auc[1]:.2f} | balanced-accuracy 95% CI {ci_bal[0]:.2f}-{ci_bal[1]:.2f}  (scan-level bootstrap)")

cm = confusion_matrix(y, pred, labels=list(range(K)))
tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
rep = pd.DataFrame({"precision": tp / np.maximum(tp + fp, 1), "recall(sensitivity)": tp / np.maximum(tp + fn, 1), "specificity": tn / np.maximum(tn + fp, 1),
                    "f1": 2 * tp / np.maximum(2 * tp + fp + fn, 1), "auc_ovr": [roc_auc_score(y == c, P[:, c]) for c in range(K)],
                    "avg_precision": [average_precision_score(y == c, P[:, c]) for c in range(K)], "support": cm.sum(1)}, index=CL).round(3)
print("\nper-class:\n", rep.to_string()); print("\nconfusion matrix (rows = true, cols = predicted):\n", pd.DataFrame(cm, index=CL, columns=CL).to_string())
pd.Series(overall).to_csv(f"{OUT}/overall_metrics.csv", header=["value"]); rep.to_csv(f"{OUT}/per_class_metrics.csv")

# ---------------- per-fold ----------------
if "fold" in d.columns:
    rows = []
    for f, g in d.groupby("fold"):
        Pf, yf = g[[f"p_{c}" for c in CL]].values, g.label.map({c: i for i, c in enumerate(CL)}).values
        rows.append(dict(fold=f, n=len(g), accuracy=accuracy_score(yf, Pf.argmax(1)), balanced_acc=balanced_accuracy_score(yf, Pf.argmax(1)),
                         macro_f1=f1_score(yf, Pf.argmax(1), average="macro", zero_division=0), macro_auc=macro_auc(yf, Pf)))
    fold = pd.DataFrame(rows).round(3); print("\nper-fold:\n", fold.to_string(index=False)); print("mean:", fold.drop(columns=["fold", "n"]).mean().round(3).to_dict())
    fold.to_csv(f"{OUT}/per_fold_metrics.csv", index=False)

# ---------------- within-source (OASIS-only classes) ----------------
W = [CL.index(c) for c in WITHIN if c in CL]
if len(W) >= 2:
    keep = np.isin(y, W); Pw = P[keep][:, W]; Pw = Pw / Pw.sum(1, keepdims=True); yw = np.array([W.index(v) for v in y[keep]]); pw = Pw.argmax(1)
    print(f"\nWITHIN-SOURCE ({' / '.join(CL[i] for i in W)}, n={keep.sum()}): accuracy {accuracy_score(yw, pw):.3f} | balanced acc {balanced_accuracy_score(yw, pw):.3f} | "
          f"macro-F1 {f1_score(yw, pw, average='macro', zero_division=0):.3f} | macro-AUC {macro_auc(yw, Pw):.3f}")

# ---------------- plots ----------------
fig, axs = plt.subplots(1, 2, figsize=(10.5, 4.3))                                       # confusion matrices
for a, (M, t, f) in zip(axs, [(cm, "Counts", "d"), (cm / np.maximum(cm.sum(1, keepdims=True), 1), "Row-normalised (recall)", ".2f")]):
    a.imshow(M, cmap="Blues", vmin=0, vmax=M.max()); a.grid(False)
    for i in range(K):
        for j in range(K): a.text(j, i, format(M[i, j], f), ha="center", va="center", color="white" if M[i, j] > M.max() * .55 else INK)
    a.set_xticks(range(K)); a.set_xticklabels(lab, rotation=25); a.set_yticks(range(K)); a.set_yticklabels(lab); a.set_xlabel("predicted"); a.set_ylabel("true"); a.set_title(t, loc="left")
fin(fig, "confusion_matrix")

fig, ax = plt.subplots(figsize=(5.8, 5))                                                 # ROC
for c in range(K):
    fpr, tpr, _ = roc_curve(y == c, P[:, c]); ax.plot(fpr, tpr, color=COL[c], lw=2, label=f"{lab[c]} (AUC {rep.auc_ovr.iloc[c]:.2f})")
ax.plot([0, 1], [0, 1], color=MUTED, ls="--", lw=1); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate"); ax.set_aspect("equal")
ax.set_title(f"ROC one-vs-rest (macro-AUC {overall['macro_auc_ovr']:.2f})", loc="left"); ax.legend(loc="lower right"); fin(fig, "roc_curves")

fig, ax = plt.subplots(figsize=(5.8, 5))                                                 # precision-recall
for c in range(K):
    pr_, rc_, _ = precision_recall_curve(y == c, P[:, c]); ax.plot(rc_, pr_, color=COL[c], lw=2, label=f"{lab[c]} (AP {rep.avg_precision.iloc[c]:.2f})")
    ax.axhline((y == c).mean(), color=COL[c], lw=.8, ls=":")
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0, 1.02); ax.set_title("Precision-recall (dotted = chance level per class)", loc="left"); ax.legend(loc="lower left"); fin(fig, "precision_recall_curves")

fig, ax = plt.subplots(figsize=(8.5, 3.9)); w = .2; xs = np.arange(K)                    # per-class bars
for k, (m, cc) in enumerate([("precision", COL[0]), ("recall(sensitivity)", COL[1]), ("specificity", COL[2]), ("f1", COL[3])]):
    bb = ax.bar(xs + (k - 1.5) * w, rep[m], w * .92, color=cc, label=m.split("(")[0])
    for b, v in zip(bb, rep[m]): ax.text(b.get_x() + b.get_width() / 2, v + .01, f"{v:.2f}", ha="center", fontsize=7, color=MUTED)
ax.set_xticks(xs); ax.set_xticklabels(lab); ax.set_ylim(0, 1.15); ax.set_title("Per-class precision / recall / specificity / F1", loc="left")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(.5, -.12)); ax.grid(axis="x", visible=False); fin(fig, "per_class_metrics")

fig, ax = plt.subplots(figsize=(7.5, 3.6)); names = ["accuracy", "balanced_accuracy", "macro_f1", "macro_auc_ovr", "mcc", "cohen_kappa"]   # overall bars
bb = ax.bar(range(len(names)), [overall[n] for n in names], color=COL[0], width=.6)
for b, n in zip(bb, names): ax.text(b.get_x() + b.get_width() / 2, overall[n] + .01, f"{overall[n]:.2f}", ha="center", color=INK)
ax.set_xticks(range(len(names))); ax.set_xticklabels(["accuracy", "balanced\nacc", "macro-F1", "macro-AUC", "MCC", "kappa"]); ax.set_ylim(0, 1.1)
ax.axhline(1 / K, color=MUTED, ls=":", lw=1); ax.text(len(names) - .5, 1 / K + .01, "chance accuracy", ha="right", color=MUTED, fontsize=8)
ax.set_title("Overall metrics (pooled out-of-fold)", loc="left"); ax.grid(axis="x", visible=False); fin(fig, "overall_metrics")

if "fold" in d.columns:
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    for m, cc, nm in [("macro_auc", COL[0], "macro-AUC"), ("balanced_acc", COL[1], "balanced acc"), ("macro_f1", COL[2], "macro-F1")]:
        ax.plot(fold.fold, fold[m], "o-", color=cc, lw=1.6, ms=6, label=nm)
    ax.set_xticks(fold.fold); ax.set_xlabel("fold"); ax.set_ylim(0, 1.05); ax.set_title("Fold-to-fold stability", loc="left"); ax.legend(ncol=3); fin(fig, "fold_stability")
print("\nsaved to", OUT, sorted(os.listdir(OUT)))

n = 60 scans | classes: ['normal', 'mci', 'alzheimers', 'stroke']

accuracy               0.783
balanced_accuracy      0.783
macro_precision        0.798
macro_recall           0.783
macro_f1               0.778
weighted_f1            0.778
macro_auc_ovr          0.925
macro_avg_precision    0.783
mcc                    0.721
cohen_kappa            0.711
log_loss               0.523

macro-AUC 95% CI 0.88-0.97 | balanced-accuracy 95% CI 0.69-0.87  (scan-level bootstrap)

per-class:
             precision  recall(sensitivity)  specificity     f1  auc_ovr  avg_precision  support
normal          1.000                0.933        1.000  0.966    0.997          0.992       15
mci             0.600                0.400        0.911  0.480    0.836          0.575       15
alzheimers      0.591                0.867        0.800  0.703    0.870          0.574       15
stroke          1.000                0.933        1.000  0.966    0.997          0.992       15

confusion matrix (rows = true, 

In [13]:
import os, shutil
from IPython.display import FileLink, display

src = "/kaggle/working/results_mc/metrics_plots"
assert os.path.isdir(src), "run metrics_plots.py first"
shutil.make_archive("/kaggle/working/metrics_plots", "zip", src)
print(round(os.path.getsize("/kaggle/working/metrics_plots.zip") / 1e6, 1), "MB")
display(FileLink("metrics_plots.zip"))      # click the link it prints

0.3 MB


/kaggle/working/metrics_plots.zip